### Imports

In [1]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess

import math

### Data Processing

In [2]:
raw_ghcnd = pd.read_csv(r'../data/raw/GHCND_1980_to_2026.csv')
raw_ghcnd["DATE"] = pd.to_datetime(raw_ghcnd["DATE"])

raw_ghcnd.info()

<class 'pandas.DataFrame'>
RangeIndex: 17093 entries, 0 to 17092
Data columns (total 86 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   STATION          17093 non-null  str           
 1   NAME             17093 non-null  str           
 2   LATITUDE         17093 non-null  float64       
 3   LONGITUDE        17093 non-null  float64       
 4   ELEVATION        17093 non-null  float64       
 5   DATE             17093 non-null  datetime64[us]
 6   ACMH             5783 non-null   float64       
 7   ACMH_ATTRIBUTES  5783 non-null   str           
 8   ACSH             5783 non-null   float64       
 9   ACSH_ATTRIBUTES  5783 non-null   str           
 10  AWND             15629 non-null  float64       
 11  AWND_ATTRIBUTES  15629 non-null  str           
 12  FMTM             10180 non-null  float64       
 13  FMTM_ATTRIBUTES  10180 non-null  str           
 14  PGTM             10783 non-null  float64       
 

C:\Users\Moral\AppData\Local\Temp\ipykernel_23064\1601894922.py:1: DtypeWarning: Columns (7: ACMH_ATTRIBUTES, 9: ACSH_ATTRIBUTES, 13: FMTM_ATTRIBUTES, 19: PSUN_ATTRIBUTES, 25: TAVG_ATTRIBUTES, 31: TSUN_ATTRIBUTES, 33: WDF1_ATTRIBUTES, 39: WDFG_ATTRIBUTES, 41: WESD_ATTRIBUTES, 43: WSF1_ATTRIBUTES, 49: WSFG_ATTRIBUTES, 57: WT04_ATTRIBUTES, 59: WT05_ATTRIBUTES, 61: WT06_ATTRIBUTES, 63: WT07_ATTRIBUTES, 67: WT09_ATTRIBUTES, 69: WT10_ATTRIBUTES, 71: WT11_ATTRIBUTES, 73: WT13_ATTRIBUTES, 75: WT14_ATTRIBUTES, 77: WT15_ATTRIBUTES, 79: WT16_ATTRIBUTES, 81: WT18_ATTRIBUTES, 83: WT21_ATTRIBUTES, 85: WV03_ATTRIBUTES) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_ghcnd = pd.read_csv(r'../data/raw/GHCND_1980_to_2026.csv')


In [3]:
def persistence_ghcnd_dataset(raw_ghcnd):
    updated_df = raw_ghcnd.set_index("DATE")

    TEMP_FEATURES = [
        "TMAX", "TMIN",
        "TMAX_lag1", "TMIN_lag1"
    ]

    # Create Lag Feature
    updated_df["TMAX_lag1"] = updated_df["TMAX"].shift(1)
    updated_df["TMIN_lag1"] = updated_df["TMIN"].shift(1)

    # Split Data
    updated_df = updated_df[1:]
    train_df = updated_df[(1980 <= updated_df.index.year) & (updated_df.index.year <= 2010)][TEMP_FEATURES]
    val_df = updated_df[(2011 <= updated_df.index.year) & (updated_df.index.year <= 2017)][TEMP_FEATURES]
    test_df = updated_df[(2018 <= updated_df.index.year) & (updated_df.index.year <= 2025)][TEMP_FEATURES]

    # Interpolate Data
    train_df.interpolate(method="time", inplace=True)
    val_df.interpolate(method="time", inplace=True)
    test_df.interpolate(method="time", inplace=True)

    # Apply Shifts and Take Out First 6 Value
    return train_df, val_df, test_df

def climatological_ghcnd_dataset(raw_ghcnd):
    updated_df = raw_ghcnd.set_index("DATE")

    # Add Values Fourier Calendar Values
    TEMP_FEATURES = [
        "TMAX", "TMIN",
    ]
    
    # Split Data
    train_df = updated_df[(1980 <= updated_df.index.year) & (updated_df.index.year <= 2010)][TEMP_FEATURES]
    val_df = updated_df[(2011 <= updated_df.index.year) & (updated_df.index.year <= 2017)][TEMP_FEATURES]
    test_df = updated_df[(2018 <= updated_df.index.year) & (updated_df.index.year <= 2025)][TEMP_FEATURES]

    # Interpolate Data
    train_df.interpolate(method="time", inplace=True)
    val_df.interpolate(method="time", inplace=True)
    test_df.interpolate(method="time", inplace=True)

    # Apply Shifts and Take Out First 6 Value
    return train_df, val_df, test_df

### Baseline Models

In [4]:
persistence_train_df, persistence_val_df, persistence_test_df = persistence_ghcnd_dataset(raw_ghcnd)
climatological_train_df, climatological_val_df, climatological_test_df = climatological_ghcnd_dataset(raw_ghcnd) # For Climatology as above removes the first 6 days of 1980

print(f"Persistence:")
print(f"- Train Size: {len(persistence_train_df)}")
print(f"- Val Size: {len(persistence_val_df)}")
print(f"- Test Size: {len(persistence_test_df)}")
print(f"\nClimatological:")
print(f"- Train Size: {len(climatological_train_df)}")
print(f"- Val Size: {len(climatological_val_df)}")
print(f"- Test Size: {len(climatological_test_df)}")

Persistence:
- Train Size: 11322
- Val Size: 2557
- Test Size: 2922

Climatological:
- Train Size: 11323
- Val Size: 2557
- Test Size: 2922


#### Metrics
- Metrics to Measure: (MAE, RMSE, Mean error, Baseline_skill)

In [5]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
def mean_error(true_y:np.ndarray, pred_y:np.ndarray):
    return (true_y - pred_y).mean(axis=0)

#### Persistence

In [6]:
class PersistenceBaseline():
    def __init__(self, train_df=None):
        pass

    def predict(self, input_df):
        output_df = input_df[["TMAX_lag1", "TMIN_lag1"]].rename(columns={"TMAX_lag1": "predicted_TMAX", "TMIN_lag1": "predicted_TMIN"})
        return output_df

In [7]:
persistence_model = PersistenceBaseline()

persistence_train_output = persistence_model.predict(persistence_train_df)
persistence_train_metrics = {
    "MAE":  mean_absolute_error(persistence_train_df[["TMAX", "TMIN"]].values, persistence_train_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(persistence_train_df[["TMAX", "TMIN"]].values, persistence_train_output.values, multioutput="raw_values"),
    "ME":   mean_error(persistence_train_df[["TMAX", "TMIN"]].values, persistence_train_output.values)
}

persistence_val_output = persistence_model.predict(persistence_val_df)
persistence_val_metrics = {
    "MAE":  mean_absolute_error(persistence_val_df[["TMAX", "TMIN"]].values, persistence_val_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(persistence_val_df[["TMAX", "TMIN"]].values, persistence_val_output.values, multioutput="raw_values"),
    "ME":   mean_error(persistence_val_df[["TMAX", "TMIN"]].values, persistence_val_output.values)
}

persistence_test_output = persistence_model.predict(persistence_test_df)
persistence_test_metrics = {
    "MAE":  mean_absolute_error(persistence_test_df[["TMAX", "TMIN"]].values, persistence_test_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(persistence_test_df[["TMAX", "TMIN"]].values, persistence_test_output.values, multioutput="raw_values"),
    "ME":   mean_error(persistence_test_df[["TMAX", "TMIN"]].values, persistence_test_output.values)
}

In [8]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- MAE: {persistence_train_metrics["MAE"]}")
print(f"- RMSE: {persistence_train_metrics["RMSE"]}")
print(f"- ME: {persistence_train_metrics["ME"]}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- MAE: {persistence_val_metrics["MAE"]}")
print(f"- RMSE: {persistence_val_metrics["RMSE"]}")
print(f"- ME: {persistence_val_metrics["ME"]}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- MAE: {persistence_test_metrics["MAE"]}")
print(f"- RMSE: {persistence_test_metrics["RMSE"]}")
print(f"- ME: {persistence_test_metrics["ME"]}\n")


Train Metrics: (TMAX/TMIN)
- MAE: [3.04531002 3.46422893]
- RMSE: [4.36767405 4.93737499]
- ME: [0.00167815 0.00123653]

Validation Metrics: (TMAX/TMIN)
- MAE: [2.94798592 3.08486508]
- RMSE: [4.18182746 4.38345808]
- ME: [-0.00312867 -0.00312867]

Test Metrics: (TMAX/TMIN)
- MAE: [2.90349076 3.10198494]
- RMSE: [4.139797   4.37882846]
- ME: [-0.00479124 -0.00342231]



#### Climatological

In [9]:
class ClimatologicalBaseline():
    def __init__(self, train_df):
        tmp_df = train_df.copy(deep=True)
        tmp_df["DATE_OF_YEAR"] = train_df.index.strftime('%j').astype(int)
        self.climatological_model = tmp_df.groupby(by="DATE_OF_YEAR")[["TMAX", "TMIN"]].mean()

    def predict(self, input_df):
        tmp_df = input_df.copy(deep=True)
        tmp_df["DATE_OF_YEAR"] = tmp_df.index.strftime('%j').astype(int)
        return tmp_df[["DATE_OF_YEAR"]].join(self.climatological_model, on="DATE_OF_YEAR")[["TMAX", "TMIN"]].rename(columns={"TMAX": "predicted_TMAX", "TMIN": "predicted_TMIN"})

In [10]:
climatological_model = ClimatologicalBaseline(climatological_train_df)

climatological_train_output = climatological_model.predict(climatological_train_df)
climatological_train_metrics = {
    "MAE":  mean_absolute_error(climatological_train_df[["TMAX", "TMIN"]].values, climatological_train_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(climatological_train_df[["TMAX", "TMIN"]].values, climatological_train_output.values, multioutput="raw_values"),
    "ME":   mean_error(climatological_train_df[["TMAX", "TMIN"]].values, climatological_train_output.values)
}

climatological_val_output = climatological_model.predict(climatological_val_df)
climatological_val_metrics = {
    "MAE":  mean_absolute_error(climatological_val_df[["TMAX", "TMIN"]].values, climatological_val_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(climatological_val_df[["TMAX", "TMIN"]].values, climatological_val_output.values, multioutput="raw_values"),
    "ME":   mean_error(climatological_val_df[["TMAX", "TMIN"]].values, climatological_val_output.values)
}

climatological_test_output = climatological_model.predict(climatological_test_df)
climatological_test_metrics = {
    "MAE":  mean_absolute_error(climatological_test_df[["TMAX", "TMIN"]].values, climatological_test_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(climatological_test_df[["TMAX", "TMIN"]].values, climatological_test_output.values, multioutput="raw_values"),
    "ME":   mean_error(climatological_test_df[["TMAX", "TMIN"]].values, climatological_test_output.values)
}

In [11]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- MAE: {climatological_train_metrics["MAE"]}")
print(f"- RMSE: {climatological_train_metrics["RMSE"]}")
print(f"- ME: {climatological_train_metrics["ME"]}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- MAE: {climatological_val_metrics["MAE"]}")
print(f"- RMSE: {climatological_val_metrics["RMSE"]}")
print(f"- ME: {climatological_val_metrics["ME"]}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- MAE: {climatological_test_metrics["MAE"]}")
print(f"- RMSE: {climatological_test_metrics["RMSE"]}")
print(f"- ME: {climatological_test_metrics["ME"]}\n")

Train Metrics: (TMAX/TMIN)
- MAE: [3.89905787 4.75531533]
- RMSE: [5.33290177 6.46322832]
- ME: [-1.16844527e-15  3.36853608e-15]

Validation Metrics: (TMAX/TMIN)
- MAE: [4.05221593 5.00005677]
- RMSE: [5.33703567 6.57116074]
- ME: [0.99674518 2.56314103]

Test Metrics: (TMAX/TMIN)
- MAE: [4.33331125 5.49668808]
- RMSE: [5.51272404 6.93480245]
- ME: [2.04475503 3.57136076]



#### Comparing Climatology with Persistence

In [12]:
def baseline_skill(baseline_metrics, pred_metrics):
    return 1 - (pred_metrics['MAE']/baseline_metrics['MAE'])

In [13]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_train_metrics, persistence_train_metrics)}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_val_metrics, persistence_val_metrics)}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_test_metrics, persistence_test_metrics)}\n")

Train Metrics: (TMAX/TMIN)
- Baseline: [0.2189626  0.27150384]

Validation Metrics: (TMAX/TMIN)
- Baseline: [0.27250029 0.38303399]

Test Metrics: (TMAX/TMIN)
- Baseline: [0.32996026 0.43566291]



### ML Models

#### Linear Regression

In [14]:
from sklearn.linear_model import LinearRegression

In [15]:
TRAIN_MEAN = raw_ghcnd.set_index("DATE")[["TMAX", "TMIN"]][(1980 <= raw_ghcnd.set_index("DATE").index.year) & (raw_ghcnd.set_index("DATE").index.year <= 2010)].interpolate(method="time").mean()
TRAIN_STD = raw_ghcnd.set_index("DATE")[["TMAX", "TMIN"]][(1980 <= raw_ghcnd.set_index("DATE").index.year) & (raw_ghcnd.set_index("DATE").index.year <= 2010)].interpolate(method="time").std()

In [16]:
# Calculate temp deltas and ways to convert and revert them
def linear_ghcnd_dataset(raw_ghcnd):
    updated_df = raw_ghcnd.set_index("DATE")
    updated_df["DATE_OF_YEAR"] = updated_df.index.map(
        lambda x: int(x.strftime('%j'))
    )

    # Add Values Fourier Calendar Values
    updated_df["FOURIER_SIN"] = updated_df.index.map(
        lambda x: math.sin(2*math.pi*(int(x.strftime('%j')) - 1)/365)
    )

    updated_df["FOURIER_COS"] = updated_df.index.map(
        lambda x: math.sin(2*math.pi*(int(x.strftime('%j')) - 1)/365)
    )

    # Normalize Temperature
    updated_df[["TMAX", "TMIN"]] = (updated_df[["TMAX", "TMIN"]] - TRAIN_MEAN)/TRAIN_STD
    
    TEMP_FEATURES = [
        "TMAX", "TMIN",
        "TMAX_lag1", "TMIN_lag1", 
        "TMAX_lag2", "TMIN_lag2", 
        "TMAX_lag3", "TMIN_lag3", 
        "TMAX_lag4", "TMIN_lag4", 
        "TMAX_lag5", "TMIN_lag5",
        "TMAX_lag6", "TMIN_lag6",
    ]
    PRCP_FEATURES = [
        "PRCP_lag1",
        "PRCP_lag2",
        "PRCP_lag3",
        "PRCP_lag4",
        "PRCP_lag5",
        "PRCP_lag6",
    ]
    FOURIER_FEATURES = [
        "FOURIER_SIN", "FOURIER_COS",
        "FOURIER_SIN_lag1", "FOURIER_COS_lag1",
        "FOURIER_SIN_lag2", "FOURIER_COS_lag2",
        "FOURIER_SIN_lag3", "FOURIER_COS_lag3",
        "FOURIER_SIN_lag4", "FOURIER_COS_lag4",
        "FOURIER_SIN_lag5", "FOURIER_COS_lag5",
        "FOURIER_SIN_lag6", "FOURIER_COS_lag6",
    ]

    updated_df["TMAX_lag1"] = updated_df["TMAX"].shift(1)
    updated_df["TMIN_lag1"] = updated_df["TMIN"].shift(1)
    updated_df["PRCP_lag1"] = updated_df["PRCP"].shift(1)
    updated_df["FOURIER_SIN_lag1"] = updated_df["FOURIER_SIN"].shift(1)
    updated_df["FOURIER_COS_lag1"] = updated_df["FOURIER_COS"].shift(1)

    updated_df["TMAX_lag2"] = updated_df["TMAX"].shift(2)
    updated_df["TMIN_lag2"] = updated_df["TMIN"].shift(2)
    updated_df["PRCP_lag2"] = updated_df["PRCP"].shift(2)
    updated_df["FOURIER_SIN_lag2"] = updated_df["FOURIER_SIN"].shift(2)
    updated_df["FOURIER_COS_lag2"] = updated_df["FOURIER_COS"].shift(2)

    updated_df["TMAX_lag3"] = updated_df["TMAX"].shift(3)
    updated_df["TMIN_lag3"] = updated_df["TMIN"].shift(3)
    updated_df["PRCP_lag3"] = updated_df["PRCP"].shift(3)
    updated_df["FOURIER_SIN_lag3"] = updated_df["FOURIER_SIN"].shift(3)
    updated_df["FOURIER_COS_lag3"] = updated_df["FOURIER_COS"].shift(3)

    updated_df["TMAX_lag4"] = updated_df["TMAX"].shift(4)
    updated_df["TMIN_lag4"] = updated_df["TMIN"].shift(4)
    updated_df["PRCP_lag4"] = updated_df["PRCP"].shift(4)
    updated_df["FOURIER_SIN_lag4"] = updated_df["FOURIER_SIN"].shift(4)
    updated_df["FOURIER_COS_lag4"] = updated_df["FOURIER_COS"].shift(4)

    updated_df["TMAX_lag5"] = updated_df["TMAX"].shift(5)
    updated_df["TMIN_lag5"] = updated_df["TMIN"].shift(5)
    updated_df["PRCP_lag5"] = updated_df["PRCP"].shift(5)
    updated_df["FOURIER_SIN_lag5"] = updated_df["FOURIER_SIN"].shift(5)
    updated_df["FOURIER_COS_lag5"] = updated_df["FOURIER_COS"].shift(5)

    updated_df["TMAX_lag6"] = updated_df["TMAX"].shift(6)
    updated_df["TMIN_lag6"] = updated_df["TMIN"].shift(6)
    updated_df["PRCP_lag6"] = updated_df["PRCP"].shift(6)
    updated_df["FOURIER_SIN_lag6"] = updated_df["FOURIER_SIN"].shift(6)
    updated_df["FOURIER_COS_lag6"] = updated_df["FOURIER_COS"].shift(6)
    
    # Split Data
    ALL_FEATURES = TEMP_FEATURES + FOURIER_FEATURES
    updated_df = updated_df[6:]
    train_df = updated_df[(1980 <= updated_df.index.year) & (updated_df.index.year <= 2010)][ALL_FEATURES]
    val_df = updated_df[(2011 <= updated_df.index.year) & (updated_df.index.year <= 2017)][ALL_FEATURES]
    test_df = updated_df[(2018 <= updated_df.index.year) & (updated_df.index.year <= 2025)][ALL_FEATURES]

    # Interpolate Data
    train_df.interpolate(method="time", inplace=True)
    val_df.interpolate(method="time", inplace=True)
    test_df.interpolate(method="time", inplace=True)

    # Apply Shifts and Take Out First 6 Value
    return train_df, val_df, test_df

In [19]:
linear_train_df, linear_val_df, linear_test_df = linear_ghcnd_dataset(raw_ghcnd)

In [32]:
linear_train_X, linear_train_y = linear_train_df.drop(columns=["TMAX", "TMIN"]).values, linear_train_df[["TMAX", "TMIN"]].values
linear_val_X, linear_val_y = linear_val_df.drop(columns=["TMAX", "TMIN"]).values, linear_val_df[["TMAX", "TMIN"]].values
linear_test_X, linear_test_y = linear_test_df.drop(columns=["TMAX", "TMIN"]).values, linear_test_df[["TMAX", "TMIN"]].values

In [33]:
linear_model = LinearRegression()
linear_model.fit(linear_train_X, linear_train_y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](2, 26)","[[ 0.74, 0.06,-0.09,...,-6.71, 5.25, 5.25], [ 0.44, 0.62,-0.17,...,-5.58, 1.62, 1.62]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float64](2,)","[0.,0.]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,26
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(19)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](26,)","[359.52,246. , 97.88,..., 0. , 0. , 0. ]"


In [38]:
linear_train_normalized_output = linear_model.predict(linear_train_X)
linear_val_normalized_output = linear_model.predict(linear_val_X)
linear_test_normalized_output = linear_model.predict(linear_test_X)

linear_train_unnormalized_output = linear_train_normalized_output*TRAIN_STD.values + TRAIN_MEAN.values
linear_val_unnormalized_output = linear_val_normalized_output*TRAIN_STD.values + TRAIN_MEAN.values
linear_test_unnormalized_output = linear_test_normalized_output*TRAIN_STD.values + TRAIN_MEAN.values

linear_train_unnormalized_y = linear_train_y*TRAIN_STD.values + TRAIN_MEAN.values
linear_val_unnormalized_y = linear_val_y*TRAIN_STD.values + TRAIN_MEAN.values
linear_test_unnormalized_y = linear_test_y*TRAIN_STD.values + TRAIN_MEAN.values

In [39]:
linear_train_metrics = {
    "MAE":  mean_absolute_error(linear_train_unnormalized_output, linear_train_unnormalized_y, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(linear_train_unnormalized_output, linear_train_unnormalized_y, multioutput="raw_values"),
    "ME":   mean_error(linear_train_unnormalized_output, linear_train_unnormalized_y)
}
linear_val_metrics = {
    "MAE":  mean_absolute_error(linear_val_unnormalized_output, linear_val_unnormalized_y, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(linear_val_unnormalized_output, linear_val_unnormalized_y, multioutput="raw_values"),
    "ME":   mean_error(linear_val_unnormalized_output, linear_val_unnormalized_y)
}
linear_test_metrics = {
    "MAE":  mean_absolute_error(linear_test_unnormalized_output, linear_test_unnormalized_y, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(linear_test_unnormalized_output, linear_test_unnormalized_y, multioutput="raw_values"),
    "ME":   mean_error(linear_test_unnormalized_output, linear_test_unnormalized_y)
}

In [40]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- MAE: {linear_train_metrics["MAE"]}")
print(f"- RMSE: {linear_train_metrics["RMSE"]}")
print(f"- ME: {linear_train_metrics["ME"]}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- MAE: {linear_val_metrics["MAE"]}")
print(f"- RMSE: {linear_val_metrics["RMSE"]}")
print(f"- ME: {linear_val_metrics["ME"]}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- MAE: {linear_test_metrics["MAE"]}")
print(f"- RMSE: {linear_test_metrics["RMSE"]}")
print(f"- ME: {linear_test_metrics["ME"]}\n")

Train Metrics: (TMAX/TMIN)
- MAE: [2.8008832  2.97036223]
- RMSE: [3.93051561 4.02858692]
- ME: [-1.31849408e-16  2.51141729e-17]

Validation Metrics: (TMAX/TMIN)
- MAE: [2.7959608  2.75664778]
- RMSE: [3.79391377 3.71616483]
- ME: [-0.3784104  -0.78909337]

Test Metrics: (TMAX/TMIN)
- MAE: [2.83568938 2.85838875]
- RMSE: [3.80612659 3.76407515]
- ME: [-0.65775671 -0.90659457]



In [41]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_train_metrics, linear_train_metrics)}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_val_metrics, linear_val_metrics)}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_test_metrics, linear_test_metrics)}\n")

Train Metrics: (TMAX/TMIN)
- Baseline: [0.28165129 0.37535957]

Validation Metrics: (TMAX/TMIN)
- Baseline: [0.31001683 0.4486767 ]

Test Metrics: (TMAX/TMIN)
- Baseline: [0.34560681 0.47997982]



In [42]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(persistence_train_metrics, linear_train_metrics)}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(persistence_val_metrics, linear_val_metrics)}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(persistence_test_metrics, linear_test_metrics)}\n")

Train Metrics: (TMAX/TMIN)
- Baseline: [0.08026336 0.1425618 ]

Validation Metrics: (TMAX/TMIN)
- Baseline: [0.05156915 0.106396  ]

Test Metrics: (TMAX/TMIN)
- Baseline: [0.02335168 0.07852913]



#### XGBoost

In [ ]:
from xgboost import XGBRegressor

In [ ]:
def xgboost_ghcnd_dataset(raw_ghcnd):
    updated_df = raw_ghcnd.set_index("DATE")
    updated_df["DATE_OF_YEAR"] = updated_df.index.map(
        lambda x: int(x.strftime('%j'))
    )

    # Add Values Fourier Calendar Values
    updated_df["FOURIER_SIN"] = updated_df.index.map(
        lambda x: math.sin(2*math.pi*(int(x.strftime('%j')) - 1)/365)
    )

    updated_df["FOURIER_COS"] = updated_df.index.map(
        lambda x: math.sin(2*math.pi*(int(x.strftime('%j')) - 1)/365)
    )

    # Normalize Temperature
    updated_df[["TMAX", "TMIN"]] = (updated_df[["TMAX", "TMIN"]] - TRAIN_MEAN)/TRAIN_STD
    
    TEMP_FEATURES = [
        "TMAX", "TMIN",
        "TMAX_lag1", "TMIN_lag1", 
        "TMAX_lag2", "TMIN_lag2", 
        "TMAX_lag3", "TMIN_lag3", 
        "TMAX_lag4", "TMIN_lag4", 
        "TMAX_lag5", "TMIN_lag5",
        "TMAX_lag6", "TMIN_lag6",
    ]
    PRCP_FEATURES = [
        "PRCP_lag1",
        "PRCP_lag2",
        "PRCP_lag3",
        "PRCP_lag4",
        "PRCP_lag5",
        "PRCP_lag6",
    ]
    FOURIER_FEATURES = [
        "FOURIER_SIN", "FOURIER_COS",
        "FOURIER_SIN_lag1", "FOURIER_COS_lag1",
        "FOURIER_SIN_lag2", "FOURIER_COS_lag2",
        "FOURIER_SIN_lag3", "FOURIER_COS_lag3",
        "FOURIER_SIN_lag4", "FOURIER_COS_lag4",
        "FOURIER_SIN_lag5", "FOURIER_COS_lag5",
        "FOURIER_SIN_lag6", "FOURIER_COS_lag6",
    ]

    updated_df["TMAX_lag1"] = updated_df["TMAX"].shift(1)
    updated_df["TMIN_lag1"] = updated_df["TMIN"].shift(1)
    updated_df["PRCP_lag1"] = updated_df["PRCP"].shift(1)
    updated_df["FOURIER_SIN_lag1"] = updated_df["FOURIER_SIN"].shift(1)
    updated_df["FOURIER_COS_lag1"] = updated_df["FOURIER_COS"].shift(1)

    updated_df["TMAX_lag2"] = updated_df["TMAX"].shift(2)
    updated_df["TMIN_lag2"] = updated_df["TMIN"].shift(2)
    updated_df["PRCP_lag2"] = updated_df["PRCP"].shift(2)
    updated_df["FOURIER_SIN_lag2"] = updated_df["FOURIER_SIN"].shift(2)
    updated_df["FOURIER_COS_lag2"] = updated_df["FOURIER_COS"].shift(2)

    updated_df["TMAX_lag3"] = updated_df["TMAX"].shift(3)
    updated_df["TMIN_lag3"] = updated_df["TMIN"].shift(3)
    updated_df["PRCP_lag3"] = updated_df["PRCP"].shift(3)
    updated_df["FOURIER_SIN_lag3"] = updated_df["FOURIER_SIN"].shift(3)
    updated_df["FOURIER_COS_lag3"] = updated_df["FOURIER_COS"].shift(3)

    updated_df["TMAX_lag4"] = updated_df["TMAX"].shift(4)
    updated_df["TMIN_lag4"] = updated_df["TMIN"].shift(4)
    updated_df["PRCP_lag4"] = updated_df["PRCP"].shift(4)
    updated_df["FOURIER_SIN_lag4"] = updated_df["FOURIER_SIN"].shift(4)
    updated_df["FOURIER_COS_lag4"] = updated_df["FOURIER_COS"].shift(4)

    updated_df["TMAX_lag5"] = updated_df["TMAX"].shift(5)
    updated_df["TMIN_lag5"] = updated_df["TMIN"].shift(5)
    updated_df["PRCP_lag5"] = updated_df["PRCP"].shift(5)
    updated_df["FOURIER_SIN_lag5"] = updated_df["FOURIER_SIN"].shift(5)
    updated_df["FOURIER_COS_lag5"] = updated_df["FOURIER_COS"].shift(5)

    updated_df["TMAX_lag6"] = updated_df["TMAX"].shift(6)
    updated_df["TMIN_lag6"] = updated_df["TMIN"].shift(6)
    updated_df["PRCP_lag6"] = updated_df["PRCP"].shift(6)
    updated_df["FOURIER_SIN_lag6"] = updated_df["FOURIER_SIN"].shift(6)
    updated_df["FOURIER_COS_lag6"] = updated_df["FOURIER_COS"].shift(6)
    
    # Split Data
    ALL_FEATURES = TEMP_FEATURES + FOURIER_FEATURES
    updated_df = updated_df[6:]
    train_df = updated_df[(1980 <= updated_df.index.year) & (updated_df.index.year <= 2010)][ALL_FEATURES]
    val_df = updated_df[(2011 <= updated_df.index.year) & (updated_df.index.year <= 2017)][ALL_FEATURES]
    test_df = updated_df[(2018 <= updated_df.index.year) & (updated_df.index.year <= 2025)][ALL_FEATURES]

    # Interpolate Data
    train_df.interpolate(method="time", inplace=True)
    val_df.interpolate(method="time", inplace=True)
    test_df.interpolate(method="time", inplace=True)

    # Apply Shifts and Take Out First 6 Value
    return train_df, val_df, test_df

In [ ]:
xgboost_train_df, xgboost_val_df, xgboost_test_df = xgboost_ghcnd_dataset(raw_ghcnd)

#### RNN